# 무엇을 잊게 만들어야 하는가

`M_o` (`m_o/M_o.pt`에 있는 모델)는 현재 100개의 ImageNet 클래스를 모두 정확하게 분류합니다. 여러분이 잊게 만들어야 할 10개 클래스도 마찬가지입니다.

이 노트북은 그 10개 클래스가 무엇인지, 어떻게 생겼는지 보여주고, `M_o`가 아직 아무것도 잊지 않았다는 것을 확인시켜 줍니다. 바로 그것이 `unlearn.py`에서 여러분이 풀어야 할 문제입니다.

In [ ]:
import json
import random

import torch
from PIL import Image
import matplotlib.pyplot as plt

from imagenet_vit import ViTWrapper
from train_ft import eval_transform

d = torch.load("splits/student_split.pt", weights_only=True)
meta, sp = d["meta"], d["splits"]
wnids, root = meta["wnids"], meta["root"]

fw = json.load(open("es/es_imagenet_mo/forget10.json"))
forget_labels = sorted(wnids.index(w) for w in fw["wnid"])

print(f"전체 {len(wnids)}개 클래스 중 잊어야 할 {len(forget_labels)}개 클래스:")
for w, n in zip(fw["wnid"], fw["name"]):
    print(f"  클래스 {wnids.index(w):3d}   {w}   {n}")

## 어떤 이미지들인가

각 forget 클래스마다 샘플 이미지 하나씩을 `released` split에서 가져와 보여줍니다. `released`는 `unlearn.py`에서 여러분이 사용할 수 있는 바로 그 학습 데이터 풀입니다.

In [ ]:
by_class = {}
for p, l in sp["released"]:
    if l in set(forget_labels):
        by_class.setdefault(l, []).append(p)

random.seed(0)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, l in zip(axes.flat, forget_labels):
    p = random.choice(by_class[l])
    img = Image.open(f"{root}/{p}").convert("RGB")
    name = fw["name"][fw["wnid"].index(wnids[l])]
    ax.imshow(img)
    ax.set_title(f"{name}\n(class {l})", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## M_o는 이미 이 클래스들을 잊었을까? (아니오)

채점 서버가 사용하는 것과 동일한 로더(`ViTWrapper`에 `strict=True`로 로드)를 사용해서, forget 클래스당 샘플 하나씩에 대해 실제로 추론해 봅니다.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ViTWrapper(num_classes=100, pretrained=False, drop_path_rate=0.0, in_model_norm=False)
sd = torch.load("m_o/M_o.pt", map_location="cpu", weights_only=True)
model.load_state_dict(sd["model"], strict=True)
model = model.eval().to(device)

tf = eval_transform(model.data_config)

random.seed(1)
samples = [(l, random.choice(by_class[l])) for l in forget_labels]
imgs = torch.stack([tf(Image.open(f"{root}/{p}").convert("RGB")) for _, p in samples]).to(device)

with torch.no_grad():
    preds = model(imgs).argmax(1).cpu().tolist()

correct = 0
for (true_l, _), pred_l in zip(samples, preds):
    true_name = fw["name"][fw["wnid"].index(wnids[true_l])]
    hit = pred_l == true_l
    correct += hit
    mark = "여전히 기억함" if hit else "이 이미지는 틀림"
    print(f"  정답={true_name:16s} (클래스 {true_l:3d})   M_o 예측: 클래스 {pred_l:3d}   {mark}")

print(f"\nM_o는 잊어야 할 클래스들에 대해 {len(samples)}개 중 {correct}개를 맞혔습니다.")
print("이것이 출발점입니다. unlearn.py를 열어 TODO 블록을 채우세요.")

## 다음 단계

- `unlearn.py`를 여세요. `load_mo`, retain/forget 데이터로더(`utils/data.py`), config/seed 설정, 채점 서버가 요구하는 저장 포맷까지 전부 미리 연결되어 있습니다. `TODO` 블록에 본인의 방법을 구현하면 됩니다.

- 점수는 **두 가지**를 함께 봅니다.
  1. **정확도 기반 (AUS)** — forget 클래스를 얼마나 잘 지웠는지 + retain 클래스 성능을 얼마나 잘 보존했는지
  2. **표현 변화 기반 (RUS_o)** — forget 표현은 원본 `M_o`에서 멀어지고 retain 표현은 `M_o`와 얼마나 그대로인지

  따라서 forget 클래스의 logit만 가려서 숨기고 내부 표현은 그대로 두는 방식은, 정확도상으로는 완벽해 보여도 RUS에서 매우 낮은 점수를 받습니다. 자세한 지표 정의는 `README.md`의 "평가 지표" 절을 참고하세요.

- 구현이 끝나면:
  ```bash
  python unlearn.py --config configs/unlearn.yaml
  python validate_submission.py --ckpt models/experiment-001.pt
  python score_model.py models/experiment-001.pt
  ```
  검증이 통과하면 랜딩 페이지의 제출 폼으로 제출하세요.